[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20collection/wikipedia_election_scraper.ipynb)

# Wikipedia scraper: 2024 general election

**DA2402 · Dr. Arun B Ayyar**

We want a table of which party contested how many seats in which state, and won how many.
Source: [2024 Indian general election](https://en.wikipedia.org/wiki/2024_Indian_general_election).
The page is static HTML so `pd.read_html` handles the parsing. Most of the work is finding the
right tables and cleaning them.

## Fetch the page

Wikipedia wants a User-Agent that says who you are. Fetch once, keep the HTML in memory.

In [1]:
import requests
import pandas as pd
from io import StringIO

URL = "https://en.wikipedia.org/wiki/2024_Indian_general_election"
UA = {"User-Agent": "DA2402-course-notebook/1.0 (IIT Madras BS DSAI; educational use)"}

html = requests.get(URL, headers=UA, timeout=30).text
print(f"{len(html):,} characters of HTML")

2,060,189 characters of HTML


## Parse the tables

`read_html` gives one DataFrame per `<table>` on the page.

In [2]:
tables = pd.read_html(StringIO(html))
print(f"{len(tables)} tables on the page")

72 tables on the page


## Finding the right tables

The page has 72 tables and the order changes when the article is edited, so don't hard-code table
numbers. The party tables all have a `Seats contested` column. Four match: NDA, two INDIA bloc
tables (seat sharing + contests outside the pact), and other notable parties, which has no
`Seats won` column.

In [3]:
def colnames(t):
    """Column labels as flat lowercase strings (handles MultiIndex headers)."""
    if isinstance(t.columns, pd.MultiIndex):
        return [" ".join(str(x) for x in c).lower() for c in t.columns]
    return [str(c).lower() for c in t.columns]

matches = [i for i, t in enumerate(tables)
           if any("seats contested" in c for c in colnames(t))]
for i in matches:
    has_won = any("seats won" in c for c in colnames(tables[i]))
    print(f"table {i}: {tables[i].shape}, has a 'Seats won' column: {has_won}")

table 5: (60, 7), has a 'Seats won' column: True
table 6: (76, 7), has a 'Seats won' column: True
table 7: (60, 7), has a 'Seats won' column: True
table 8: (83, 5), has a 'Seats won' column: False


## Raw table

Things that need cleaning: an unnamed colour-swatch column, footnote markers on numbers
(`75[58]`), party-total columns repeated on every row (merged cells), and a `Total` row at the
bottom that doubles any sum.

In [4]:
raw = tables[matches[0]]
raw.head(4)

,Party,Party.1,State/UT,Seats contested,Seats contested.1,Seats won,Seats won.1
0,NaN,Bharatiya Janata Party,Uttar Pradesh,75[58],441[59],33,240
1,NaN,Bharatiya Janata Party,West Bengal,42,441[59],12,240
2,NaN,Bharatiya Janata Party,Madhya Pradesh,29,441[59],29,240
3,NaN,Bharatiya Janata Party,Maharashtra,28,441[59],9,240


In [5]:
raw.tail(2)  # the Total row

,Party,Party.1,State/UT,Seats contested,Seats contested.1,Seats won,Seats won.1
58,NaN,Independent,Tamil Nadu,1,1,0,0
59,Total,Total,Total,541,541,293,293


## Cleaning

Name the columns, strip footnotes, drop the `Total` row but keep its number for a check later.
Alliance labels come from the contents: the table with BJP is NDA, tables with INC are INDIA
bloc. INDIA spans two tables so its check total accumulates.

In [6]:
import re

def clean(t):
    """One alliance table -> (per-state rows, wikipedia's own total-won figure)."""
    t = t.copy()
    t.columns = ["swatch", "party", "state", "contested", "contested_total", "won", "won_total"]
    strip = lambda s: re.sub(r"\[\d+\]", "", str(s))          # "75[58]" -> "75"
    for c in ["contested", "won", "won_total"]:
        t[c] = pd.to_numeric(t[c].map(strip), errors="coerce")
    total_row = t[t["party"].eq("Total")]
    wiki_total = int(total_row["won"].iloc[0]) if len(total_row) else None
    t = t[t["party"].ne("Total")].dropna(subset=["state"])
    return t[["party", "state", "contested", "won"]], wiki_total

def alliance_of(t):
    parties = set(t.iloc[:, 1].astype(str))
    if "Bharatiya Janata Party" in parties:   return "NDA"
    if "Indian National Congress" in parties: return "INDIA bloc"
    return "Unaligned"

frames, checks = [], {}
for i in matches:
    if not any("seats won" in c for c in colnames(tables[i])):
        continue                                   # no-won-column table: step 6
    tidy, wiki_total = clean(tables[i])
    tidy.insert(0, "alliance", alliance_of(tables[i]))
    frames.append(tidy)
    label = tidy["alliance"].iloc[0]
    checks[label] = checks.get(label, 0) + wiki_total   # INDIA bloc spans two tables

results = pd.concat(frames, ignore_index=True)
print(f"{len(results)} party-state rows, {results['party'].nunique()} parties")
results.head(8)

193 party-state rows, 52 parties


,alliance,party,state,contested,won
0,NDA,Bharatiya Janata Party,Uttar Pradesh,75,33
1,NDA,Bharatiya Janata Party,West Bengal,42,12
2,NDA,Bharatiya Janata Party,Madhya Pradesh,29,29
3,NDA,Bharatiya Janata Party,Maharashtra,28,9
4,NDA,Bharatiya Janata Party,Gujarat,26,25
5,NDA,Bharatiya Janata Party,Karnataka,25,17
6,NDA,Bharatiya Janata Party,Rajasthan,25,14
7,NDA,Bharatiya Janata Party,Tamil Nadu,23,0


## Other parties table, and the issue with it

This table only records seats contested. Some of these parties did win (YSRCP 4, AIMIM, Akali
Dal etc) but the article does not give their wins state-wise, so `won` is set to `<NA>`. Filling
0 would be wrong. The header is a two-level MultiIndex and there is an aggregate row
("Unrecognised parties") to drop.

In [7]:
nowins = next(tables[i] for i in matches
              if not any("seats won" in c for c in colnames(tables[i])))
nowins = nowins.copy()
nowins.columns = ["swatch", "party", "state", "contested", "contested_total"][:len(nowins.columns)]
nowins["contested"] = pd.to_numeric(
    nowins["contested"].map(lambda s: re.sub(r"\[\d+\]", "", str(s))), errors="coerce")
nowins = (nowins[~nowins["party"].isin(["Total", "Unrecognised parties"])]
          .dropna(subset=["party", "state", "contested"])
          [["party", "state", "contested"]])
nowins.insert(0, "alliance", "Other")
nowins["won"] = pd.NA          # not zero: some of these parties won

results = pd.concat([results, nowins], ignore_index=True)
results = results.astype({"contested": "Int64", "won": "Int64"})
print(f"{len(results)} rows, {results['party'].nunique()} parties, "
      f"{results['state'].nunique()} states/UTs")

259 rows, 78 parties, 40 states/UTs


## Checks

BJP won 240 and INC 99, so assert those. Per-state sums should match the Total rows we dropped.
NDA + INDIA = 527 of 543; the other 16 seats are independents and the `<NA>` parties.

One quirk: DMK and KMDK share a merged totals cell (KMDK contested on the DMK symbol), so their
totals column is off by one. The per-state values are correct, which is why we sum states.

In [8]:
by_party = results.groupby("party")[["contested", "won"]].sum()
assert by_party.loc["Bharatiya Janata Party", "won"] == 240
assert by_party.loc["Indian National Congress", "won"] == 99

for alliance, wiki_total in checks.items():
    ours = results.loc[results["alliance"].eq(alliance), "won"].sum()
    flag = "ok" if ours == wiki_total else "MISMATCH"
    print(f"{alliance:11s} our sum {ours:3d}   wikipedia's Total row {wiki_total:3d}   {flag}")

alliance_seats = int(results["won"].sum())          # <NA> rows are skipped
print(f"\n{alliance_seats} of 543 seats; the other {543 - alliance_seats} are "
      f"independents and the <NA> parties")

NDA         our sum 293   wikipedia's Total row 293   ok
INDIA bloc  our sum 234   wikipedia's Total row 234   ok

527 of 543 seats; the other 16 are independents and the <NA> parties


## Final table

One row per (party, state). Summaries are one-liners from here.

In [9]:
results.sort_values(["alliance", "party", "contested"],
                    ascending=[True, True, False], ignore_index=True)

,alliance,party,state,contested,won
0,INDIA bloc,Aam Aadmi Party,Punjab,13,3
1,INDIA bloc,Aam Aadmi Party,Delhi,4,0
2,INDIA bloc,Aam Aadmi Party,Gujarat,2,0
3,INDIA bloc,Aam Aadmi Party,Assam,2,0
4,INDIA bloc,Aam Aadmi Party,Haryana,1,0
...,...,...,...,...,...
254,Other,United Democratic Party,Meghalaya,1,<NA>
255,Other,Uttarakhand Kranti Dal,Uttarakhand,3,<NA>
256,Other,Voice of the People Party,Meghalaya,1,<NA>
257,Other,YSR Congress Party,Andhra Pradesh,25,<NA>


In [10]:
# party totals
summary = (results.groupby(["alliance", "party"])
           .agg(states=("state", "nunique"),
                contested=("contested", "sum"),
                won=("won", lambda s: s.sum(min_count=1)))   # keep <NA>, don't sum to 0
           .sort_values("won", ascending=False))
summary.head(15)

states  \
alliance   party                                                      
NDA        Bharatiya Janata Party                                33   
INDIA bloc Indian National Congress                              36   
           Samajwadi Party                                        4   
           All India Trinamool Congress                           4   
           Dravida Munnetra Kazhagam                              1   
NDA        Telugu Desam Party                                     1   
           Janata Dal (United)                                    1   
INDIA bloc Shiv Sena (Uddhav Balasaheb Thackeray)                 1   
           Nationalist Congress Party (Sharadchandra Pawar)       3   
NDA        Shiv Sena                                              1   
           Lok Janshakti Party (Ram Vilas)                        1   
INDIA bloc Rashtriya Janata Dal                                   2   
           Communist Party of India (Marxist)                    15   
           Indian Union Muslim League                             2   
           Aam Aadmi Party                                        5   

                                                             contested  won  
alliance   party                                                             
NDA        Bharatiya Janata Party                                  441  240  
INDIA bloc Indian National Congress                                328   99  
           Samajwadi Party                                          71   37  
           All India Trinamool Congress                             48   29  
           Dravida Munnetra Kazhagam                                21   21  
NDA        Telugu Desam Party                                       17   16  
           Janata Dal (United)                                      16   12  
INDIA bloc Shiv Sena (Uddhav Balasaheb Thackeray)                   21    9  
           Nationalist Congress Party (Sharadchandra Pawar)         12    8  
NDA        Shiv Sena                                                15    7  
           Lok Janshakti Party (Ram Vilas)                           5    5  
INDIA bloc Rashtriya Janata Dal                                     24    4  
           Communist Party of India (Marxist)                       52    4  
           Indian Union Muslim League                                3    3  
           Aam Aadmi Party                                          22    3

In [11]:
# one state; change the name and rerun
results[results["state"].eq("Tamil Nadu")].sort_values("won", ascending=False).head(10)

,alliance,party,state,contested,won
104,INDIA bloc,Dravida Munnetra Kazhagam,Tamil Nadu,21,21
75,INDIA bloc,Indian National Congress,Tamil Nadu,9,9
97,INDIA bloc,Communist Party of India (Marxist),Tamil Nadu,2,2
111,INDIA bloc,Communist Party of India,Tamil Nadu,2,2
127,INDIA bloc,Viduthalai Chiruthaigal Katchi,Tamil Nadu,2,2
124,INDIA bloc,Indian Union Muslim League,Tamil Nadu,1,1
105,INDIA bloc,Kongunadu Makkal Desia Katchi,Tamil Nadu,1,1
132,INDIA bloc,Marumalarchi Dravida Munnetra Kazhagam,Tamil Nadu,1,1
58,NDA,Independent,Tamil Nadu,1,0
7,NDA,Bharatiya Janata Party,Tamil Nadu,23,0


In [12]:
results.to_csv("election_2024_party_state.csv", index=False)
print(f"saved {len(results)} rows to election_2024_party_state.csv")

saved 259 rows to election_2024_party_state.csv
